# Community Fish Detector × FiftyOne — Demo Notebook

A hands-on tour of [FiftyOne](https://docs.voxel51.com) built on the
[Community Fish Detector (CFD)](https://github.com/filippovarini/community-fish-detector)
and the [Community Fish Detection Dataset](https://lila.science/datasets/community-fish-detection-dataset/).

**The story:** a single `fish` class trained on ~17 wildly different source datasets —
murky brackish water, tropical reef GoPro footage, freshwater underwater-video billabongs,
and 32 cm lab zebrafish tanks. The dataset preserves a per-image `dataset` provenance
field, so we run the model, evaluate it, and then **slice performance by source
environment** to expose exactly where a general-purpose detector breaks down.

**What this notebook does**
- Streams a small, curated, license-safe subset of the dataset (no bulk download) and loads
  it into FiftyOne with ground-truth boxes and per-image source provenance.
- Runs the RF-DETR CFD model, evaluates it, and slices performance by source environment.
- Computes embeddings, similarity, uniqueness, and label-mistake signals.
- Persists **every demo moment as a named _saved view_.** Saved views store the view
  *definition* (filters, sorts, limits), not copies of the data, so they re-evaluate live
  against the dataset. Recreating any demo state is one click in the App's view-selector
  dropdown (the bookmark icon, top-left), or one `dataset.load_saved_view(name)` call.
- Tolerates sources whose annotations are polygon segmentations (no `bbox`) or RLE masks.

> Defaults touch only CC-BY / CC0 / CDLA / Apache / MIT sources, so the curated subset is
> safe to show publicly. NC-restricted sources are skipped.

> **Two datasets, one notebook.** Part 1 (below) uses CFD for in-the-wild *detection* and
> the domain-gap story. **Part 2** (at the end) adds **Fish-Vista** — museum specimen images
> for species classification and pixel-level *trait segmentation* — to show FiftyOne handling
> a very different fish-imagery modality: segmentation masks and long-tail taxonomy.

## 0. Prerequisites & environment

This notebook assumes **only that you have a Python 3.10–3.12 virtual environment** — it does
*not* assume FiftyOne (or anything else) is already installed. The first code cell installs
everything fresh into the running kernel.

Create (or activate) a virtual environment and launch Jupyter from inside it, so this
notebook's kernel is that environment:

```bash
python3 -m venv .venv                 # Python 3.10-3.12
source .venv/bin/activate             # macOS / Linux
# .venv\Scripts\activate              # Windows (PowerShell / cmd)
pip install jupyter
jupyter lab                           # or: jupyter notebook
```

The next cell does a **clean uninstall + fresh install** of FiftyOne, the model, and helpers.
Do this once, then **restart the kernel** before running anything below so the freshly
installed packages load cleanly. (A clean reinstall also fixes a common symptom after a
FiftyOne version bump: a blank App screen caused by a stale browser bundle that no longer
matches the server.)

> Using a fresh virtual environment is the most reliable setup. If pip prints conflict
> warnings about *unrelated* packages already installed elsewhere in the environment, they're
> generally safe to ignore for this demo — nothing here depends on them.

In [ ]:
# FRESH SETUP — assumes only a venv, not that FiftyOne is installed.
# Run this once, then RESTART THE KERNEL before running the rest of the notebook.

# 1) Clean out any previous install of the packages this notebook uses.
#    (pip harmlessly skips any that aren't present.)
%pip uninstall -y fiftyone fiftyone-brain fiftyone-db fiftyone-desktop voxel51-eta \
    rfdetr supervision umap-learn huggingface_hub

# 2) Fresh install.
%pip install --upgrade pip
%pip install fiftyone rfdetr supervision umap-learn requests pillow huggingface_hub

# 3) Verify (run AFTER restarting the kernel).
import fiftyone as fo
print("FiftyOne:", fo.__version__)

## 1. Configuration

Everything tunable lives here. `TARGET_SOURCES` is a *wish list* matched (case-insensitive
substring) against the real `dataset` values in the metadata, so a rename or typo degrades
gracefully. We pick a spread of environments on purpose, to make the domain gap visible.


In [ ]:
from pathlib import Path

WORK_DIR    = Path("./cfd_fiftyone_demo").resolve()
IMAGES_DIR  = WORK_DIR / "images"
META_DIR    = WORK_DIR / "metadata"
WEIGHTS_DIR = WORK_DIR / "weights"
for d in (IMAGES_DIR, META_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Port for the FiftyOne App. 5151 is FiftyOne's default; 5252 avoids clashing with anything
# already using the default port on your machine. Change if needed.
APP_PORT = 5252

# Wish list spanning very different visual domains (matched by substring).
TARGET_SOURCES = [
    "coralscapes",     # tropical reef, diver GoPro (Apache-2.0)
    "roboflow",        # web-sourced fish photos (CC0)
    "deepfish",        # Australian seafloor habitats (MIT) - segmentation-derived boxes
    "brackish",        # murky brackish water, Denmark (CC-BY-SA)
    "zebrafish",       # lab tank, top-down (CC-BY-4.0)
    "mit_sea_grant",   # river herring, freshwater video frames (CDLA)
    "puget",           # nearshore aquaculture video frames (CDLA)
]

IMAGES_PER_SOURCE = 70      # cap per source; lower for a faster run
ONLY_VAL_SPLIT    = True    # use the reference validation split only
PREFER_BOXED      = True    # bias sampling toward images that contain fish

MODEL_CHOICE   = "nano"     # one of: "nano", "small", "medium"
CONF_THRESHOLD = 0.25       # lower than repo's 0.3 to surface more false positives

DATASET_NAME = "community-fish-detector-demo"

LILA_BASE    = "https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset"
META_ZIP_URL = f"{LILA_BASE}/community_fish_detection_dataset.json.zip"

MODEL_WEIGHTS = {
    "nano":   ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "cfd-2026.02.02-rf-detr-nano/community-fish-detector-2026.02.02-rf-detr-nano-640.pth", 640),
    "small":  ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "2026.05.13-release/fish-detector-rf-detr-small-1024-2026.06.06-checkpoint_16.stripped.pth", 1024),
    "medium": ("https://github.com/filippovarini/community-fish-detector/releases/download/"
               "2026.05.13-release/fish-detector-rf-detr-medium-1024-2026.03.24-checkpoint_11.stripped.pth", 1024),
}
print("Workspace:", WORK_DIR)

## 2. Download & parse the COCO metadata

The full dataset is >1.9M images / >935k boxes, but the **metadata** is a single COCO JSON.
We download the zip once and load it.

> **Memory note:** the unzipped JSON loads to ~2–4 GB in RAM — fine on most machines. On a
> low-memory machine, swap `json.load` for a streaming parser (`ijson`); nothing else changes.

Beyond standard COCO fields, each image carries `dataset` (source), `is_train` (split), and
`original_data_source`.


In [ ]:
import json, zipfile, requests

def download(url, dest: Path, desc=""):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"✓ cached: {dest.name}")
        return dest
    print(f"↓ downloading {desc or dest.name} ...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total, done = int(r.headers.get("content-length", 0)), 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()
    return dest

meta_zip = download(META_ZIP_URL, META_DIR / "cfd_metadata.json.zip", desc="COCO metadata")
with zipfile.ZipFile(meta_zip) as z:
    json_name = next(n for n in z.namelist() if n.endswith(".json"))
    with z.open(json_name) as f:
        coco = json.load(f)

print("images:", len(coco["images"]), "| annotations:", len(coco["annotations"]),
      "| categories:", [c["name"] for c in coco["categories"]])

In [ ]:
from collections import defaultdict

images_by_id = {img["id"]: img for img in coco["images"]}
anns_by_image = defaultdict(list)
for a in coco["annotations"]:
    anns_by_image[a["image_id"]].append(a)

by_source = defaultdict(list)
for img in coco["images"]:
    by_source[img.get("dataset", "unknown")].append(img)

print(f"{'source':<40}{'total':>10}{'val':>10}{'boxed':>10}")
print("-" * 70)
for src in sorted(by_source, key=lambda s: -len(by_source[s])):
    imgs = by_source[src]
    n_val = sum(1 for i in imgs if not i.get("is_train", True))
    n_box = sum(1 for i in imgs if anns_by_image.get(i["id"]))
    print(f"{src:<40}{len(imgs):>10}{n_val:>10}{n_box:>10}")

## 3. Curate a small, diverse subset

Match the wish list to real source names, sample up to `IMAGES_PER_SOURCE` validation
images per source (preferring images that contain fish), and download just those.


In [ ]:
import random, requests
from concurrent.futures import ThreadPoolExecutor
random.seed(51)

real_sources = list(by_source.keys())

def resolve(wish):
    w = wish.lower()
    return [s for s in real_sources if w in s.lower()]

chosen_sources = []
for wish in TARGET_SOURCES:
    for s in resolve(wish):
        if s not in chosen_sources:
            chosen_sources.append(s)
if not chosen_sources:
    print("No wish-list sources matched; using the largest available sources.")
    chosen_sources = sorted(by_source, key=lambda s: -len(by_source[s]))[:5]
print("Selected sources:", chosen_sources)

# A few images referenced in the manifest are missing from the public bucket. We probe
# candidates with a cheap HEAD first and keep only ones that exist, so the download step
# below never 404s. OVERSAMPLE gives the probe extra candidates to backfill any misses.
OVERSAMPLE = 2
_sess = requests.Session()

def _exists(img):
    url = f"{LILA_BASE}/{img['file_name']}"
    try:
        r = _sess.head(url, timeout=20, allow_redirects=True)
        if r.status_code == 405:  # some blobs disallow HEAD; probe 1 byte instead
            r = _sess.get(url, timeout=20, stream=True, headers={"Range": "bytes=0-0"})
        return img["id"], r.status_code < 400
    except Exception:
        return img["id"], False

selected_images = []
for src_name in chosen_sources:
    pool = by_source[src_name]
    if ONLY_VAL_SPLIT:
        val = [i for i in pool if not i.get("is_train", True)]
        pool = val or pool
    if PREFER_BOXED:
        boxed   = [i for i in pool if anns_by_image.get(i["id"])]
        unboxed = [i for i in pool if not anns_by_image.get(i["id"])]
        random.shuffle(boxed); random.shuffle(unboxed)
        pool = boxed + unboxed
    else:
        random.shuffle(pool)

    candidates = pool[:IMAGES_PER_SOURCE * OVERSAMPLE]
    with ThreadPoolExecutor(max_workers=16) as ex:
        exists = dict(ex.map(_exists, candidates))
    kept = [img for img in candidates if exists.get(img["id"])][:IMAGES_PER_SOURCE]
    selected_images.extend(kept)
    skipped = len(candidates) - sum(exists.values())
    note = f"  (skipped {skipped} missing blobs)" if skipped else ""
    print(f"  {src_name:<40} kept {len(kept):>3}{note}")

print("Total images to fetch:", len(selected_images))

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_image(img):
    file_name = img["file_name"]
    local = IMAGES_DIR / file_name
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists() and local.stat().st_size > 0:
        return img["id"], local, None
    url = f"{LILA_BASE}/{file_name}"
    try:
        with requests.get(url, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 16):
                    f.write(chunk)
        return img["id"], local, None
    except Exception as e:
        return img["id"], None, str(e)

local_paths, errors = {}, []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(fetch_image, img) for img in selected_images]
    for i, fut in enumerate(as_completed(futures), 1):
        iid, path, err = fut.result()
        (errors.append((iid, err)) if err else local_paths.__setitem__(iid, path))
        if i % 25 == 0 or i == len(futures):
            print(f"\r  fetched {i}/{len(futures)} (errors: {len(errors)})", end="")
print()
if errors:
    print("Some images failed and will be skipped:", errors[:3], "...")
print("Images available locally:", len(local_paths))

### Build the FiftyOne dataset (with a robust box converter)

COCO boxes are `[x, y, w, h]` absolute pixels; FiftyOne wants them normalized to `[0, 1]`.
Some CFD sources (e.g. DeepFish, F4K) came from **segmentation masks**, so a subset of
annotations have no `bbox`. The converter below:

- uses `bbox` when present,
- derives a tight box from a **polygon** `segmentation` when `bbox` is missing,
- skips **RLE** masks (dict-encoded) and degenerate boxes,

and reports how many annotations it skipped, so you can spot a source that's mask-only.


In [ ]:
from PIL import Image

if DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(DATASET_NAME)
dataset = fo.Dataset(DATASET_NAME, persistent=True)

skipped_anns = 0

def _bbox_from_polygon(seg):
    """Tight [x, y, w, h] (abs px) from a COCO polygon segmentation; None if unusable."""
    if not isinstance(seg, list) or not seg:
        return None  # RLE dict or empty -> skip
    xs, ys = [], []
    for poly in seg:
        if not isinstance(poly, (list, tuple)):
            return None
        xs.extend(poly[0::2]); ys.extend(poly[1::2])
    if not xs or not ys:
        return None
    return [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]

def coco_to_fo_boxes(img_record, W, H):
    global skipped_anns
    dets = []
    for a in anns_by_image.get(img_record["id"], []):
        bbox = a.get("bbox") or _bbox_from_polygon(a.get("segmentation"))
        if not bbox or len(bbox) != 4:
            skipped_anns += 1; continue
        x, y, w, h = bbox
        if w <= 0 or h <= 0:
            skipped_anns += 1; continue
        dets.append(fo.Detection(label="fish",
                                 bounding_box=[x / W, y / H, w / W, h / H]))
    return fo.Detections(detections=dets)

samples = []
for img in selected_images:
    path = local_paths.get(img["id"])
    if path is None:
        continue
    try:
        with Image.open(path) as im:
            W, H = im.size
    except Exception:
        W, H = img.get("width"), img.get("height")
        if not (W and H):
            continue
    s = fo.Sample(filepath=str(path))
    s["ground_truth"] = coco_to_fo_boxes(img, W, H)
    s["source"] = img.get("dataset", "unknown")
    s["split"]  = "train" if img.get("is_train", True) else "val"
    s["original_data_source"] = img.get("original_data_source")
    s["n_gt"] = len(s["ground_truth"].detections)
    samples.append(s)

dataset.add_samples(samples)
dataset.compute_metadata()
print(f"Skipped {skipped_anns} annotations without a usable box (RLE / degenerate).")
print(dataset)
print("\nImages per source:", dataset.count_values("source"))

## 4. First look in the App

Color by `source` and filter `ground_truth` to feel how different these environments look,
even though every box is just "fish".


In [ ]:
# Launch on APP_PORT (not 5151) to avoid local port collisions.
# If the tab ever looks blank after a version change, hard-refresh (Cmd/Ctrl+Shift+R).
session = fo.launch_app(dataset, port=APP_PORT)
print(f"App: http://localhost:{APP_PORT}/")
session

## 5. Run the Community Fish Detector

Download the RF-DETR weights and run inference into a `predictions` field. Several warnings
are expected and harmless: the "not optimized for inference" notice only matters for GPU FP16
tensor cores, and the DINOv2 backbone / class-count messages are normal when loading a
fully-trained single-class checkpoint.


In [ ]:
weights_url, resolution = MODEL_WEIGHTS[MODEL_CHOICE]
weights_path = download(weights_url, WEIGHTS_DIR / Path(weights_url).name, desc=f"{MODEL_CHOICE} weights")
print("Weights:", weights_path, "| resolution:", resolution)

In [ ]:
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium

_MODEL_CLASSES = {"nano": RFDETRNano, "small": RFDETRSmall, "medium": RFDETRMedium}
model = _MODEL_CLASSES[MODEL_CHOICE](pretrain_weights=str(weights_path), resolution=resolution)
print("Loaded RF-DETR", MODEL_CHOICE)

In [ ]:
def sv_to_fo(det, W, H, label="fish"):
    out = []
    xyxy = det.xyxy
    confs = det.confidence if det.confidence is not None else [None] * len(xyxy)
    for (x1, y1, x2, y2), c in zip(xyxy, confs):
        out.append(fo.Detection(
            label=label,
            bounding_box=[float(x1)/W, float(y1)/H, float(x2-x1)/W, float(y2-y1)/H],
            confidence=None if c is None else float(c),
        ))
    return fo.Detections(detections=out)

with fo.ProgressBar() as pb:
    for sample in pb(dataset):
        image = Image.open(sample.filepath).convert("RGB")
        W, H = image.size
        det = model.predict(image, threshold=CONF_THRESHOLD)
        sample["predictions"] = sv_to_fo(det, W, H)
        sample["n_pred"] = len(sample["predictions"].detections)
        sample.save()

print("Predictions per image (min, max):", dataset.bounds("n_pred"))

## 6. Evaluate overall (COCO mAP)

`evaluate_detections` tags each prediction `tp`/`fp` and each missed GT box `fn` in an
`eval` field, adds sample-level `eval_tp`/`eval_fp`/`eval_fn` counts, and computes COCO mAP.
These fields are what several saved views below filter on.


In [ ]:
results = dataset.evaluate_detections(
    "predictions", gt_field="ground_truth", eval_key="eval", compute_mAP=True,
)
print(f"Overall COCO mAP: {results.mAP():.3f}\n")
results.print_report()

## 7. The domain-gap reveal: performance **per source**

The payoff. A single number hides everything; per-source mAP shows where the detector is
production-ready and where it collapses. We capture the best/worst sources to save as views.


In [ ]:
import pandas as pd
from fiftyone import ViewField as F

rows = []
for src in dataset.distinct("source"):
    sview = dataset.match(F("source") == src)
    r = sview.evaluate_detections("predictions", gt_field="ground_truth",
                                  eval_key=None, compute_mAP=True)
    rows.append({"source": src, "images": sview.count(),
                 "gt_boxes": sview.sum("n_gt"), "pred_boxes": sview.sum("n_pred"),
                 "mAP": round(r.mAP(), 3)})

df = pd.DataFrame(rows).sort_values("mAP", ascending=False).reset_index(drop=True)
best_source  = df.iloc[0]["source"]
worst_source = df.iloc[-1]["source"]
print("best:", best_source, "| worst:", worst_source)
df

## 8. Embeddings, similarity, uniqueness, mistakenness

Compute the Brain runs that the analytical saved views depend on. In the App's
**Embeddings** panel, color by `source` — environments separate into clusters, the visual
explanation for the per-source mAP spread.


In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(dataset, model="clip-vit-base32-torch",
                          brain_key="img_viz", embeddings="clip_embeddings")
fob.compute_similarity(dataset, embeddings="clip_embeddings", brain_key="img_sim")
fob.compute_uniqueness(dataset)
fob.compute_mistakenness(dataset, "predictions", label_field="ground_truth")

session.refresh()
print("Brain runs:", dataset.list_brain_runs())

## 9. Persist every demo moment as a saved view

This is the centerpiece. Each view below is stored on the (persistent) dataset by name, so:

- In the **App**, pick it from the view-selector dropdown (bookmark icon, top-left).
- In **code**, `dataset.load_saved_view(name)` or `session.view = dataset.load_saved_view(name)`.

Saved views store the *pipeline*, not the samples, so they always reflect the current data.
The helper is idempotent — re-running replaces existing views of the same name.

The last group uses `to_evaluation_patches`, which turns the view into **one crop per
true positive / false positive / false negative**. Those are the "see the failure" views:
open `eval-patches: false positives` for a wall of hallucinated fish, or
`eval-patches: missed fish` for everything the model walked past — far more legible on a
projector than error boxes buried in full scenes.

> **Single-class note.** `compute_mistakenness` leans on class *disagreement*, which barely
> exists with one class, and its `possible_missing` flag is gated at 0.95 confidence — higher
> than detection models usually reach — so those fields are often empty here. The
> missing-label views below therefore derive "confident fish, no GT box" straight from the
> evaluation's false positives at a reachable `MISSING_CONF` threshold, which actually fires.


In [ ]:
from fiftyone import ViewField as F

def save(name, view, description=""):
    if dataset.has_saved_view(name):
        dataset.delete_saved_view(name)
    dataset.save_view(name, view, description=description)
    print(f"  saved: {name:<34} ({view.count()} samples)")

# compute_mistakenness flags `possible_missing` only for FPs above a hard-coded 0.95
# confidence, which detection models rarely reach (so that field is usually empty).
# We derive the "confident fish, no GT box" signal from the evaluation instead, at a
# reachable threshold. Lower this if your model's confidences run low.
MISSING_CONF = 0.7

print("Per-source views:")
for src in dataset.distinct("source"):
    save(f"source: {src}", dataset.match(F("source") == src),
         description=f"All images from the '{src}' source dataset.")

print("\nDomain-gap views:")
save("best source (highest mAP)",
     dataset.match(F("source") == best_source),
     description=f"Highest-scoring source ({best_source}). Model does well here.")
save("worst source (lowest mAP)",
     dataset.match(F("source") == worst_source),
     description=f"Lowest-scoring source ({worst_source}). Where the detector struggles.")

print("\nError-analysis views:")
save("high-confidence false positives",
     dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") > 0.5))
            .sort_by(F("predictions.detections").length(), reverse=True),
     description="Confident detections that don't match any GT box (>0.5 conf).")
save("missed fish (false negatives)",
     dataset.filter_labels("ground_truth", F("eval") == "fn"),
     description="Ground-truth fish the model failed to detect.")
save("no detections",
     dataset.match(F("n_pred") == 0),
     description="Images where the model predicted nothing — candidate misses.")
save("crowded scenes",
     dataset.sort_by("n_pred", reverse=True).limit(25),
     description="Most detections per image — where false positives tend to hide.")

print("\nData-quality views:")
save("likely label mistakes",
     dataset.sort_by("mistakenness", reverse=True).limit(25),
     description="Highest mistakenness — probable GT annotation errors.")
save("possible missing annotations",
     dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") >= MISSING_CONF)),
     description="Confident (≥%.2f) fish predictions with no matching GT box — possible missing labels." % MISSING_CONF)
save("near-duplicate frames",
     dataset.sort_by("uniqueness").limit(25),
     description="Least-unique images — typically near-duplicate video frames.")
save("most unique / hardest",
     dataset.sort_by("uniqueness", reverse=True).limit(25),
     description="Most-unique images — rare or visually distinctive samples.")

print("\nEvaluation-patches views (one tile per error — the 'see the failure' payoff):")
try:
    eval_patches = dataset.to_evaluation_patches("eval")
    save("eval-patches: false positives",
         eval_patches.match(F("type") == "fp"),
         description="One crop per confident hallucinated fish (false positive).")
    save("eval-patches: missed fish",
         eval_patches.match(F("type") == "fn"),
         description="One crop per fish the model missed (false negative).")
    save("eval-patches: true positives",
         eval_patches.match(F("type") == "tp"),
         description="One crop per correct detection (true positive).")
except Exception as e:
    print("  (skipped patches views — not saveable on this build:", e, ")")

print("\nLabel-mistake views (spurious side — the 'missing' side is 'eval-patches: missed fish'):")
_spurious = dataset.match(F("possible_spurious") > 0).sort_by("possible_spurious", reverse=True)
if _spurious.count() > 0:
    save("possible spurious labels", _spurious,
         description="Images with GT boxes the model thinks shouldn't exist — possible over-labeling.")
else:
    print("  (no possible_spurious flags on this single-class run — skipping; use 'missed fish (false negatives)' instead)")
try:
    gt_patches = dataset.to_patches("ground_truth")
    save("label mistakes (patch view)",
         gt_patches.sort_by("ground_truth.mistakenness", reverse=True).limit(50),
         description="Top 50 ground-truth boxes ranked by mistakenness — one crop each.")
except Exception as e:
    print("  (skipped label-mistake patch view — not saveable on this build:", e, ")")

# Which sources have the most confident false positives (likely-missing labels)? Target those.
fp_conf = dataset.filter_labels("predictions", (F("eval") == "fp") & (F("confidence") >= MISSING_CONF))
missing_by_src = {s: fp_conf.match(F("source") == s).count("predictions.detections")
                  for s in dataset.distinct("source")}
top_missing_sources = [s for s, n in sorted(missing_by_src.items(), key=lambda kv: -kv[1]) if n > 0][:3]
try:
    missing_patches = (dataset.to_patches("predictions")
                       .match((F("predictions.eval") == "fp") & (F("predictions.confidence") >= MISSING_CONF))
                       .sort_by("predictions.confidence", reverse=True))
    if top_missing_sources:
        missing_patches = missing_patches.match(F("source").is_in(top_missing_sources))
    if missing_patches.count() > 0:
        save("missing labels (high-confidence)", missing_patches,
             description=("One crop per confident (≥%.2f) false positive — likely-missing labels" % MISSING_CONF)
                         + ((", worst sources: " + ", ".join(top_missing_sources)) if top_missing_sources else "")
                         + ". Review before filing upstream.")
    else:
        print("  (no confident FPs — lower MISSING_CONF to populate the missing-labels view)")
except Exception as e:
    print("  (skipped high-confidence missing-labels view:", e, ")")

print("\nAll saved views:")
for v in dataset.list_saved_views():
    print("  •", v)

## 10. Tour the saved views

Load any view by name. In a live demo you'd instead just pick these from the App dropdown.


In [ ]:
# Jump straight to the model's most embarrassing mistakes:
session.view = dataset.load_saved_view("high-confidence false positives")
print("Now showing:", "high-confidence false positives")

In [ ]:
# The domain-gap contrast, back to back:
session.view = dataset.load_saved_view("worst source (lowest mAP)")
print("Worst source in view. Swap to best with the line below.")
# session.view = dataset.load_saved_view("best source (highest mAP)")

In [ ]:
# Data-quality pass — probable label errors. Patch view = one crop per suspect box:
session.view = dataset.load_saved_view("label mistakes (patch view)")
print("Now showing one crop per suspect box, most-likely-wrong first.")
# Image-level version:  dataset.load_saved_view("likely label mistakes")
# Over-labeling side:    dataset.load_saved_view("possible spurious labels")

In [ ]:
# Missing-label bugs to file upstream: confident fish with no GT box, worst sources first.
_name = "missing labels (high-confidence)"
if dataset.has_saved_view(_name):
    session.view = dataset.load_saved_view(_name)
    print("Showing likely-missing labels in the most-affected sources.")
else:
    print("No missing-label view (no confident FPs — lower MISSING_CONF in the save cell).")

In [ ]:
# The "see the failure" payoff: a wall of individual error crops.
# Swap "false positives" <-> "missed fish" to flip between hallucinations and misses.
session.view = dataset.load_saved_view("eval-patches: false positives")
print("Now showing one crop per false positive. Try 'eval-patches: missed fish' too.")

## 11. Wrap-up & reuse

Everything persists with the dataset. In a fresh session you can skip straight to the
interesting parts:

```python
import fiftyone as fo
dataset = fo.load_dataset("community-fish-detector-demo")
session = fo.launch_app(dataset)
session.view = dataset.load_saved_view("worst source (lowest mAP)")
```

**Talking points**
- One `fish` class, many environments — the setup for a domain-gap story.
- The per-source mAP table turns one opaque number into a map of where the detector is
  production-ready and where it isn't.
- Embeddings visually explain that spread; the error and data-quality views close the loop
  from "the metric is low" to "here are the exact images to fix or relabel."
- Saved views make all of the above a one-click recreate during a live walkthrough.

**Extending**
- Swap `MODEL_CHOICE="medium"` (1024px) and compare per-source mAP to nano.
- Raise `IMAGES_PER_SOURCE` / add sources for a fuller picture.
- Add the video sources (Salmon CV, FishCLEF, F4K) as FiftyOne video datasets (some are NC).


In [ ]:
# Manage saved views / clean up.
print("Saved views:", dataset.list_saved_views())

# Delete one:        dataset.delete_saved_view("crowded scenes")
# Delete all views:  dataset.delete_saved_views()
# Delete dataset:    fo.delete_dataset(DATASET_NAME)

---
# Part 2 — Fish-Vista: specimen trait analysis

[Fish-Vista](https://huggingface.co/datasets/imageomics/fish-vista)
([paper](https://arxiv.org/abs/2407.08027) ·
[code](https://github.com/sajeedmehrab/Fish-Vista)) is a very different kind of fish
dataset from CFD: ~60K **museum specimen images** on clean white backgrounds, spanning
~1,900 species, curated from natural-history collections (GLIN, iDigBio, Morphbank) via
[Fish-AIR](https://fishair.org/). It supports species classification, trait identification,
and **pixel-level trait segmentation** of 9 anatomical structures (eye, head, barbel, and the
dorsal / adipose / pectoral / pelvic / anal / caudal fins).

Where Part 1 showed FiftyOne on *detection in the wild*, Part 2 exercises a different set of
muscles:
- **Semantic segmentation masks** rendered over each specimen (the standout visual).
- **Multi-label traits** derived directly from those masks (which anatomy is present).
- **Long-tail taxonomy** — the dataset is explicitly imbalanced; FiftyOne's distributions and
  views make the tail tangible.
- **Provenance** by source museum and owning institution.

> **Licensing:** Fish-Vista is released under **CC-BY-NC 4.0** (individual images vary); use it
> for research / non-commercial demos. We pull a **small subset per-file via `huggingface_hub`**
> — no 11.9 GB clone and no Git LFS. We deliberately avoid `datasets.load_dataset()` because the
> Hub's Arrow build currently errors on one metadata column; downloading the raw CSV + images is
> unaffected.

## 12. Configuration & install

`huggingface_hub` was installed in the Part 1 setup cell. Everything tunable for Fish-Vista
lives here. Defaults keep the run laptop-friendly.

In [ ]:
from pathlib import Path

FV_WORK   = Path("./fish_vista_demo").resolve()
FV_MASKS  = FV_WORK / "masks_norm"        # normalized single-channel masks we render
for d in (FV_WORK, FV_MASKS):
    d.mkdir(parents=True, exist_ok=True)

FV_REPO       = "imageomics/fish-vista"    # HuggingFace dataset repo
FV_SUBSET_CSV = "segmentation_data.csv"    # collation of all trait-segmentation splits
FV_N          = 300                        # number of specimens to pull (has masks)
FV_DATASET    = "fish-vista-demo"

print("Fish-Vista workspace:", FV_WORK)

## 13. Download the segmentation metadata

We grab just the segmentation CSV and the trait-id→name map. `hf_hub_download` fetches and
caches individual files on demand — no full-repo clone.

In [ ]:
import json, ast, re, pandas as pd
from huggingface_hub import HfApi, hf_hub_download

# The repo layout on `main` shifts over time, so discover the real files rather than assume
# fixed paths. Exclude the CC-BY-ND background-removal helpers ("ND_*" / "background"),
# which are whole-fish silhouettes, not the 9 anatomical trait masks we want.
api = HfApi()
repo_files = api.list_repo_files(FV_REPO, repo_type="dataset")

def _no_nd_path(f):
    fl = f.lower()
    return "nd_" not in fl and "background" not in fl

# 1) ALL segmentation split CSVs (train/test/val/data) — concatenated below to maximize the pool
seg_csvs = [f for f in repo_files if f.lower().endswith(".csv")
            and "segmentation" in f.lower() and _no_nd_path(f)]
if not seg_csvs:
    raise RuntimeError(f"No segmentation CSV found. CSVs: "
                       f"{[f for f in repo_files if f.lower().endswith('.csv')]}")

# 2) trait id->name map json
trait_map_file = next((f for f in repo_files
                       if f.lower().endswith(".json") and "trait" in f.lower() and _no_nd_path(f)), None)

# 3) the TRAIT mask directory (under segmentation_masks/, not the ND background masks)
trait_mask_pngs = [f for f in repo_files if f.lower().endswith(".png")
                   and "segmentation_masks" in f.lower() and _no_nd_path(f)]
MASK_DIR = trait_mask_pngs[0].rsplit("/", 1)[0] if trait_mask_pngs else None

print("segmentation CSVs:", seg_csvs)
print("trait map json   :", trait_map_file)
print("mask directory   :", MASK_DIR, f"({len(trait_mask_pngs)} trait masks)")

def hf_get(fname):
    return hf_hub_download(FV_REPO, fname, repo_type="dataset")

# Concatenate every available segmentation split into one pool of specimens-with-masks.
frames = []
for f in seg_csvs:
    d = pd.read_csv(hf_get(f))
    d["split"] = f.rsplit("/", 1)[-1].replace("segmentation_", "").replace(".csv", "")
    frames.append(d)
fv_df = pd.concat(frames, ignore_index=True)
if "filename" in fv_df.columns:
    fv_df = fv_df.drop_duplicates(subset="filename").reset_index(drop=True)

# Drop CC-BY-ND rows: their processed images aren't published in the repo and aren't
# redistributable. Token match (not substring) so we don't false-hit "Usage Conditions".
def _is_nd(lic):
    return "ND" in re.findall(r"[A-Za-z]+", str(lic).upper())
if "license" in fv_df.columns:
    n0 = len(fv_df)
    fv_df = fv_df[~fv_df["license"].map(_is_nd)].reset_index(drop=True)
    print(f"dropped {n0 - len(fv_df)} CC-BY-ND rows (not redistributable); {len(fv_df)} remain")

# Parse the trait map. This repo's file uses bare integer keys (e.g. {0: "Background"}),
# which is a valid Python literal but NOT strict JSON — try json first, then ast.
MASK_TARGETS = {0: "background"}
if trait_map_file:
    raw = open(hf_get(trait_map_file)).read().strip()
    parsed = None
    for loader in (json.loads, ast.literal_eval):
        try:
            parsed = loader(raw); break
        except Exception:
            continue
    if parsed:
        MASK_TARGETS = {int(k): v for k, v in parsed.items()}
    else:
        print("WARNING: could not parse trait map; masks will show numeric ids. raw:", raw[:160])
else:
    print("WARNING: no trait-map json found; masks will show numeric ids.")

print("\nspecimens with masks:", len(fv_df), "| columns:", list(fv_df.columns))
print("traits:", MASK_TARGETS)
if "source" in fv_df.columns:
    print("\nspecimens per source museum:")
    print(fv_df["source"].value_counts())

## 14. Curate a subset, then download images + masks

We sample `FV_N` rows and fetch, per specimen, the processed image and its trait-segmentation
mask. Masks are normalized to single-channel PNGs (what FiftyOne renders from `mask_path`),
and we read back which trait ids appear so we can tag each specimen with the anatomy that's
actually segmented.

In [ ]:
import os, numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download

if MASK_DIR is None:
    raise RuntimeError("No mask directory discovered in the previous cell; cannot build "
                       "the trait-segmentation dataset.")

# The segmentation CSV has no 'file_name' (repo path) column, so resolve each specimen's
# image by matching its `filename` against the repo's image listing (works whether images
# are flat under Images/ or chunked under Images/chunk_*/).
img_files = [f for f in repo_files
             if f.lower().endswith((".jpg", ".jpeg", ".png"))
             and "images" in f.lower()
             and "segmentation_masks" not in f.lower()
             and "nd_" not in f.lower() and "background" not in f.lower()]
by_base = {}
for f in img_files:
    b = os.path.basename(f)
    by_base.setdefault(b, f)
    by_base.setdefault(os.path.splitext(b)[0], f)
print("indexed", len(img_files), "repo images")

def resolve_image_repo_path(row):
    fn = str(row["filename"])
    return by_base.get(fn) or by_base.get(os.path.splitext(fn)[0])

sample_df = fv_df.sample(n=min(FV_N, len(fv_df)), random_state=51).reset_index(drop=True)

def fetch_and_prep(row):
    try:
        img_repo = resolve_image_repo_path(row)
        if img_repo is None:
            return row.name, None, None, None, f"no repo image for {row['filename']}"
        img_path = hf_hub_download(FV_REPO, img_repo, repo_type="dataset")

        base = os.path.splitext(str(row["filename"]))[0]
        raw_mask = hf_hub_download(FV_REPO, f"{MASK_DIR}/{base}.png", repo_type="dataset")

        arr = np.array(Image.open(raw_mask))
        if arr.ndim == 3:                      # collapse triplicated-gray to 2D
            arr = arr[..., 0]
        arr = arr.astype("uint8")
        norm = FV_MASKS / f"{base}.png"
        Image.fromarray(arr, mode="L").save(norm)

        present = [MASK_TARGETS[int(i)] for i in np.unique(arr)
                   if int(i) in MASK_TARGETS and int(i) != 0]
        return row.name, img_path, str(norm), present, None
    except Exception as e:
        return row.name, None, None, None, str(e)

fetched, errors = [], []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(fetch_and_prep, r) for _, r in sample_df.iterrows()]
    for i, fut in enumerate(as_completed(futures), 1):
        idx, img, mask, present, err = fut.result()
        (errors.append((idx, err)) if err else fetched.append((idx, img, mask, present)))
        if i % 20 == 0 or i == len(futures):
            print(f"\r  fetched {i}/{len(futures)} (errors: {len(errors)})", end="")
print()
if errors:
    print("Skipped (missing/unresolved):", errors[:3], "...")
print("Specimens ready:", len(fetched))

## 15. Build the Fish-Vista FiftyOne dataset

Each sample gets the trait-segmentation mask (rendered from disk), species / family / source
as classification fields, and the list of traits present in its mask. Family names in the raw
data have inconsistent casing, so we normalize them — itself a small taste of the cleanup
FiftyOne makes visible.

In [ ]:
import fiftyone as fo

if FV_DATASET in fo.list_datasets():
    fo.delete_dataset(FV_DATASET)
fvds = fo.Dataset(FV_DATASET, persistent=True)
fvds.default_mask_targets = MASK_TARGETS   # so the App shows trait names on hover
fvds.save()

fv_samples = []
for idx, img_path, mask_path, present in fetched:
    row = sample_df.loc[idx]
    s = fo.Sample(filepath=img_path)
    s["trait_mask"] = fo.Segmentation(mask_path=mask_path)
    s["species"]    = fo.Classification(label=str(row["standardized_species"]))
    s["family"]     = fo.Classification(label=str(row["family"]).strip().lower())
    s["source"]     = fo.Classification(label=str(row["source"]))
    s["owner"]      = None if pd.isna(row.get("owner")) else str(row.get("owner"))
    s["license"]    = None if pd.isna(row.get("license")) else str(row.get("license"))
    s["traits_present"] = present
    s["n_traits"]   = len(present)
    fv_samples.append(s)

fvds.add_samples(fv_samples)
fvds.compute_metadata()
print(fvds)
print("\nSpecimens per source:", fvds.count_values("source.label"))
print("Traits segmented (across subset):",
      fvds.count_values("traits_present"))

## 16. First look: trait masks in the App

Launch the App on the Fish-Vista dataset (this repoints the App from Part 1). Toggle the
`trait_mask` overlay and hover a pixel to see the trait name; color the grid by `family` to
feel the taxonomy. This is the standout Fish-Vista visual — color-coded fins, eye, and head
painted onto each specimen.

In [ ]:
# Repoints the App to the Fish-Vista dataset (same App/server, new dataset).
session = fo.launch_app(fvds, port=APP_PORT)
print(f"App: http://localhost:{APP_PORT}/")
session

## 17. Long-tail taxonomy

Fish-Vista is deliberately imbalanced. Here's the shape of that tail across the subset — a
handful of common families dominate while many species appear once.

In [ ]:
fam_counts = fvds.count_values("family.label")
sp_counts  = fvds.count_values("species.label")
singletons = [s for s, c in sp_counts.items() if c == 1]

print(f"families: {len(fam_counts)} | species: {len(sp_counts)} | singleton species: {len(singletons)}")
print("\ntop families:")
for fam, n in sorted(fam_counts.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {fam:<20} {n}")

## 18. Embeddings & uniqueness

CLIP embeddings over clean specimen photos organize largely by body plan / taxonomy — color
the Embeddings panel by `family` to see it. Uniqueness surfaces re-photographed or duplicate
specimens.

In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(fvds, model="clip-vit-base32-torch",
                          brain_key="fv_viz", embeddings="clip_emb")
fob.compute_uniqueness(fvds)
session.refresh()
print("Brain runs:", fvds.list_brain_runs())

## 19. Persist Fish-Vista demo moments as saved views

Same saved-view pattern as Part 1: each highlight becomes a one-click view in the App
dropdown or a `fvds.load_saved_view(name)` call. Trait-based views are derived from the masks
themselves, so they're always consistent with the ground truth.

In [ ]:
from fiftyone import ViewField as F

def save_fv(name, view, description=""):
    if fvds.has_saved_view(name):
        fvds.delete_saved_view(name)
    fvds.save_view(name, view, description=description)
    print(f"  saved: {name:<34} ({view.count()} samples)")

print("Provenance views:")
for src in fvds.distinct("source.label"):
    save_fv(f"fv source: {src}", fvds.match(F("source.label") == src),
            description=f"Specimens sourced from {src}.")

print("\nTaxonomy views:")
for fam, n in sorted(fam_counts.items(), key=lambda kv: -kv[1])[:5]:
    save_fv(f"fv family: {fam}", fvds.match(F("family.label") == fam),
            description=f"{fam} specimens (n={n}).")
if singletons:
    save_fv("fv rarest species (long tail)",
            fvds.match(F("species.label").is_in(singletons)),
            description=f"{len(singletons)} singleton species — the long tail of the distribution.")

print("\nTrait views (derived from the masks):")
_trait_names = [v for v in MASK_TARGETS.values() if v != "background"]
_preferred   = [t for t in _trait_names if t.lower() in ("barbel", "adipose fin")]
for t in (_preferred or _trait_names)[:2]:
    save_fv(f"fv has: {t}", fvds.match(F("traits_present").contains(t)),
            description=f"Specimens whose segmentation includes the '{t}' trait.")
save_fv("fv trait-rich specimens",
        fvds.sort_by("n_traits", reverse=True).limit(25),
        description="Most anatomical traits segmented per specimen.")

print("\nData-quality view:")
save_fv("fv near-duplicate specimens",
        fvds.sort_by("uniqueness").limit(25),
        description="Least-unique specimens — re-photographed or duplicate views.")

print("\nAll Fish-Vista saved views:")
for v in fvds.list_saved_views():
    print("  •", v)

## 21. Demo: multi-label trait queries (traits from masks)

Because `traits_present` is read straight from each segmentation mask, we get **ground-truth
multi-label attributes for free** — no extra annotation. That powers compound anatomy queries.
Try toggling a single trait in the App's mask legend to scan one structure (say, the adipose
fin) across many species — comparative anatomy as a scrollable grid.

In [ ]:
from fiftyone import ViewField as F

_traits = [v for v in MASK_TARGETS.values() if v.lower() != "background"]
print("traits available:", _traits)

def has(t):
    return F("traits_present").contains(t)

if "Barbel" in _traits:
    save_fv("fv trait: has barbel", fvds.match(has("Barbel")),
            "Barbel present — taxonomically diagnostic (e.g. catfishes, carps).")
if "Adipose fin" in _traits:
    save_fv("fv trait: has adipose fin", fvds.match(has("Adipose fin")),
            "Adipose fin present (salmonids, catfishes, characins ...).")
if "Pelvic fin" in _traits:
    save_fv("fv trait: missing pelvic fin", fvds.match(~has("Pelvic fin")),
            "No pelvic fin in the mask — truly absent, or unlabeled (a QA prompt).")
if {"Barbel", "Adipose fin"} <= set(_traits):
    save_fv("fv trait: barbel AND adipose", fvds.match(has("Barbel") & has("Adipose fin")),
            "Compound query: both barbel and adipose fin present.")
save_fv("fv trait: most complete", fvds.sort_by("n_traits", reverse=True).limit(25),
        "Most anatomical traits segmented per specimen.")

# Point the App at a compound query as a live example.
if {"Barbel", "Adipose fin"} <= set(_traits):
    session.view = fvds.load_saved_view("fv trait: barbel AND adipose")
    print("Showing specimens with BOTH a barbel and an adipose fin.")

## 22. Demo: morphology similarity search

On clean, uniform specimen photos, image embeddings organize largely by **body plan**. Color
the Embeddings panel by `family` to see taxonomy emerge as clusters; lasso a cluster to fill
the grid with lookalikes. Below we also build a similarity index and query the 25 specimens
most morphologically similar to a chosen one.

In [ ]:
import fiftyone.brain as fob

# Similarity index (reuses the CLIP embeddings computed earlier).
if not fvds.has_brain_run("fv_sim"):
    fob.compute_similarity(fvds, embeddings="clip_emb", brain_key="fv_sim")

anchor = fvds.first()
similar = fvds.sort_by_similarity(anchor.id, k=25, brain_key="fv_sim")
save_fv("fv similar to first specimen",
        similar,
        description=f"25 specimens most similar in body shape to {anchor.species.label}.")
session.view = similar
print("Anchor species:", anchor.species.label, "\u2192 showing 25 nearest by morphology.")
print("Tip: in the Embeddings panel, color by `family` and lasso a cluster.")

## 23. Demo: data-quality audit

Practical "clean before you train" workflow. We compute how much of each image the trait mask
covers, then surface suspiciously sparse masks (likely annotation problems), near-duplicate /
re-photographed specimens (via uniqueness), and per-source imaging differences. Because every
specimen carries its `license`, this also doubles as a redistribution-aware audit.

In [ ]:
import numpy as np
from PIL import Image
from fiftyone import ViewField as F

# Fraction of image pixels labeled as any trait (non-background) = rough mask coverage.
for s in fvds.iter_samples(progress=True, autosave=True):
    m = np.array(Image.open(s.trait_mask.mask_path))
    if m.ndim == 3:
        m = m[..., 0]
    s["mask_coverage"] = float((m > 0).mean())

print("mask coverage (min, max):", fvds.bounds("mask_coverage"))

save_fv("fv audit: sparse masks",
        fvds.match(F("mask_coverage") < 0.05).sort_by("mask_coverage"),
        description="Trait mask covers <5% of the image — likely empty/broken annotations.")
# per-source imaging comparison is already covered by the 'fv source: *' views.
print("\nmask coverage by source museum (mean):")
for srcm in fvds.distinct("source.label"):
    v = fvds.match(F("source.label") == srcm)
    print(f"  {srcm:<12} n={v.count():>4}  mean_coverage={v.mean('mask_coverage'):.3f}")

## 24. Demo: run the CFD detector on Fish-Vista (cross-dataset probe)

**Can Part 1's wild-fish detector work on museum specimens?** This is a large domain shift —
RF-DETR was trained on murky in-situ footage and has never seen a specimen on white. We:

1. derive a **whole-fish ground-truth box** from each specimen's trait mask (bounding box of
   all non-background pixels — head + fins bound the body tightly),
2. run the CFD model to get predicted fish boxes,
3. **evaluate** the cross-domain detection and slice it by family.

The likely outcome is the interesting part: the shift runs in the *favorable* direction — a
centered, high-contrast fish on a plain background is easier to detect than a half-occluded one
in turbid water — so expect **very high recall and confidence**. Read that as a robustness /
**auto-annotation** result: a field-trained detector pre-labeling a completely different imaging
modality with no fine-tuning. The residual signal lives in the few edge cases: any missed or
low-confidence specimens cluster on unusual body plans (rays, eels, flatfish), and multi-box
detections tend to fire on scale bars, tags, or a second specimen.


In [ ]:
import numpy as np
from PIL import Image

# Ensure the CFD RF-DETR model from Part 1 is available.
try:
    model
except NameError:
    try:
        from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium
        _cls = {"nano": RFDETRNano, "small": RFDETRSmall, "medium": RFDETRMedium}[MODEL_CHOICE]
        model = _cls(pretrain_weights=str(weights_path), resolution=resolution)
        print("Reloaded CFD model:", MODEL_CHOICE)
    except Exception as e:
        raise RuntimeError("Run Part 1's model cells first (needs `model`, `weights_path`, "
                           "`resolution`, `MODEL_CHOICE`).") from e

def mask_bbox(mask_path):
    m = np.array(Image.open(mask_path))
    if m.ndim == 3:
        m = m[..., 0]
    ys, xs = np.where(m > 0)
    if len(xs) == 0:
        return None
    H, W = m.shape[:2]
    return [float(xs.min())/W, float(ys.min())/H,
            float(xs.max()-xs.min())/W, float(ys.max()-ys.min())/H]

def sv_to_fo(det, W, H, label="fish"):
    out = []
    confs = det.confidence if det.confidence is not None else [None]*len(det.xyxy)
    for (x1, y1, x2, y2), cf in zip(det.xyxy, confs):
        out.append(fo.Detection(label=label,
            bounding_box=[float(x1)/W, float(y1)/H, float(x2-x1)/W, float(y2-y1)/H],
            confidence=None if cf is None else float(cf)))
    return fo.Detections(detections=out)

with fo.ProgressBar() as pb:
    for s in pb(fvds):
        img = Image.open(s.filepath).convert("RGB"); W, H = img.size
        gt = mask_bbox(s.trait_mask.mask_path)
        s["fish_box"] = (fo.Detections(detections=[fo.Detection(label="fish", bounding_box=gt)])
                         if gt else fo.Detections())
        det = model.predict(img, threshold=0.25)
        s["cfd_pred"] = sv_to_fo(det, W, H)
        s["cfd_n"] = len(s["cfd_pred"].detections)
        s["cfd_top_conf"] = max([d.confidence for d in s["cfd_pred"].detections], default=0.0)
        s.save()

print("CFD boxes per specimen (min, max):", fvds.bounds("cfd_n"))
print("mean top confidence:", round(fvds.mean("cfd_top_conf"), 3))

In [ ]:
from fiftyone import ViewField as F

results = fvds.evaluate_detections("cfd_pred", gt_field="fish_box",
                                   eval_key="cfd_eval", compute_mAP=True)
have_gt = fvds.match(F("fish_box.detections").length() > 0)
recall = have_gt.match(F("cfd_eval_tp") > 0).count() / max(have_gt.count(), 1)
print(f"CFD-on-Fish-Vista  mAP={results.mAP():.3f}   recall={recall:.3f}  "
      f"(found the specimen in {recall*100:.0f}% of images)")

save_fv("cfd on FV: confident hits",
        fvds.match(F("cfd_top_conf") >= 0.7).sort_by("cfd_top_conf", reverse=True),
        "Museum specimens the wild-fish detector recognizes with high confidence.")
save_fv("cfd on FV: missed specimens",
        fvds.match(F("cfd_n") == 0),
        "No detection at all — often unusual body plans (rays, eels, flatfish).")
save_fv("cfd on FV: multi-box (possible artifacts)",
        fvds.match(F("cfd_n") >= 2).sort_by("cfd_n", reverse=True),
        "2+ boxes — split detections or firing on tags / scale bars / labels.")
save_fv("cfd on FV: low-confidence morphologies",
        fvds.match(F("cfd_n") > 0).sort_by("cfd_top_conf").limit(25),
        "The 25 detected specimens the model was LEAST confident on — the body plans that "
        "look least like a swimming fish, even when all scores are high.")

session.view = fvds.load_saved_view("cfd on FV: missed specimens")
print("Showing specimens the CFD detector missed — scan for unusual morphologies.")

In [ ]:
import pandas as pd
from fiftyone import ViewField as F

# Which families does the wild-fish detector find least "fish-like"?
rows = []
for fam in fvds.distinct("family.label"):
    v = fvds.match(F("family.label") == fam)
    if v.count() < 3:
        continue
    rows.append({"family": fam, "n": v.count(),
                 "mean_top_conf": round(v.mean("cfd_top_conf"), 3),
                 "miss_rate": round(v.match(F("cfd_n") == 0).count() / v.count(), 3)})
fam_conf = pd.DataFrame(rows).sort_values("mean_top_conf").reset_index(drop=True)
print("Families the CFD detector is least confident on (top = least fish-like to the model):")
fam_conf.head(15)

In [ ]:
from fiftyone import ViewField as F

# Spotlight: the single most interesting specimen in the cross-probe — the only one where the
# detector produced an extra box (split detection, or firing on a tag / scale bar / 2nd fish).
multi = fvds.match(F("cfd_n") >= 2)
if multi.count():
    session.view = multi.sort_by("cfd_n", reverse=True)
    ex = multi.first()
    print(f"Multi-box specimen: {ex.species.label}  ({ex.cfd_n} boxes, "
          f"top conf {ex.cfd_top_conf:.2f})")
    print("Open it in the App and toggle `cfd_pred` (predicted) vs `fish_box` (mask-derived GT)")
    print("to see the near-perfect overlap plus the one spurious box.")
else:
    print("No multi-box specimens this run.")

# For any specimen: compare predicted box(es) against the mask-derived GT box in the App by
# toggling the `cfd_pred` and `fish_box` overlays. They should overlap almost exactly.
best = fvds.sort_by("cfd_top_conf", reverse=True).first()
print(f"\nHighest-confidence hit: {best.species.label} @ {best.cfd_top_conf:.3f} "
      f"— a clean example of CFD auto-annotating a specimen it was never trained on.")

## 20. Tour & wrap-up

Load any Fish-Vista view by name (or pick it from the App dropdown). In a fresh session you
can jump straight back in:

```python
import fiftyone as fo
fvds = fo.load_dataset("fish-vista-demo")
session = fo.launch_app(fvds, port=5252)
session.view = fvds.load_saved_view("fv rarest species (long tail)")
```

**Why the two datasets together tell a bigger story:** CFD is messy in-situ *detection* across
17 capture environments; Fish-Vista is clean specimen *segmentation + classification* across a
long taxonomic tail. One tool ingests both — boxes, masks, single- and multi-label — and the
same moves (evaluate, slice, embed, find mistakes, save views) apply to each. Section 24 even
runs the CFD detector *on* Fish-Vista: a field-trained model auto-labels museum specimens it
was never trained on at ~0.97 mAP / 100% recall — a robustness result that closes the loop
between the two halves.

In [ ]:
# Jump to the standout visual: specimens with the most traits segmented.
session.view = fvds.load_saved_view("fv trait-rich specimens")
print("Showing trait-rich specimens. Toggle the trait_mask overlay in the App.")

# Manage / clean up:
# print(fvds.list_saved_views())
# fo.delete_dataset(FV_DATASET)
print("Datasets on this machine:", fo.list_datasets())